In [1]:
import sys

sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn import metrics
from sklearn.metrics import auc, roc_curve, confusion_matrix, ConfusionMatrixDisplay
import os
import random
from natsort import natsorted, index_natsorted
from statistics import mode
import glob

import torch
import torchvision
from PIL import Image
import time
import copy
import tqdm
import re
import torchvision.models as models


from torch.utils.data import WeightedRandomSampler

from typing import Any, Callable, cast, Dict, List, Optional, Tuple, Union

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

torch.set_grad_enabled(False)

# Test set selection

In [17]:
test_dataset_path = '/home/trevor/repos/deep_TEE/data/semi-processed/unsupervised/test'
subject_paths = glob.glob(test_dataset_path + '/*')
clip_paths = glob.glob(test_dataset_path + '/*/*', recursive=True)

seed = 42
random.seed(seed)
random.shuffle(clip_paths)

In [ ]:
class_names = ['deep_tg_col_a', 'dta_sax', 'dta_sax_arch', 'me_2_ch', 'me_2_ch_col_m', 'me_4_ch', 'me_4_ch_col_m', 'me_4_ch_col_t', 'me_5_ch', 'me_5_ch_col', 'me_av_sax', 'me_av_sax_col', 'me_bicaval', 'me_bicaval_col', 'me_in_out', 'me_in_out_col_p', 'me_in_out_col_t', 'me_lax', 'me_lax_col_a', 'me_lax_col_m', 'me_lax_col_mv_av', 'me_mit_com', 'me_mit_com_col_m', 'other', 'tg_basal_sax', 'tg_mid_pap_sax', 'ue_asc_ao_sax']


print('Made directory for cycle_{}'.format(str(cycle).zfill(2)))
os.makedirs(f'/home/trevor/repos/deep_TEE/labels/view_predictions/test_set_annotations', exist_ok=True)

results = pd.DataFrame({'clip_path':clip_paths,})

rows = 10
cols = 10
print('Saving image snapshots')


for page in range(len(results['clip_path'])//rows + 1):
    fig, ax = plt.subplots(rows, cols, figsize=(40,rows * 3))
    for i, clip_path in enumerate(list(results['clip_path'])[page * rows:]):
        images = glob.glob(clip_path+'/*.png')

        indices = natsorted(list(torch.randint(len(images), size=(cols,))))
        for j, index in enumerate(indices):
            ax[i, j].imshow(np.array(Image.open(images[index])))
            ax[i, j].axis('off')
        ax[i, 2].set_title(f'\n{clip_path}')
        if i == rows-1:
            break
    fig.suptitle(f'{view}  page {page}', y=.92)
    plt.savefig(f'/home/trevor/repos/deep_TEE/labels/view_predictions/cycle_{str(cycle).zfill(2)}/{view}_page_{page}_cycle_{str(cycle).zfill(2)}.png')
    plt.close()
    
    # results.loc[results['prediction']==id].to_csv(f'/home/trevor/repos/deep_TEE/labels/view_predictions/cycle_{str(cycle).zfill(2)}/{view}_cycle_{str(cycle).zfill(2)}.csv', index=False)


# For moving the test studies over

In [ ]:
test_studies = pd.read_csv('/home/trevor/repos/deep_TEE/labels/500_reports_groundtruth.csv')
test_studies = test_studies.dropna()
test_studies = test_studies.drop(columns=['MRN','Report #','SURGERY_DATE','JOINED_IMPRESSION'])

In [ ]:
import shutil

In [ ]:
len(glob.glob('/data/deep_tee/semi-processed/unsupervised_2/test/*'))

484

In [ ]:
counter = 0
for study in tqdm.tqdm(test_studies['ACC_NUM']):
    if os.path.exists(f'/data/deep_tee/semi-processed/unsupervised_2/test/{int(study)}'):
        continue
    try:
        shutil.move(f'/data/deep_tee/semi-processed/unsupervised_2/train/{int(study)}', f'/data/deep_tee/semi-processed/unsupervised_2/test/{int(study)}')
    except:
        print(int(study))
        counter += 1
print(counter)

100%|██████████| 500/500 [00:00<00:00, 5642.06it/s]

31559026
35935212
31346510
32186162
41067973
34573847
31473958
31253071
31605035
31783151
31200413
31046770
36297545
34513134
34585230
43778175
16
